# 📖 Notebook 6: Child Workflows & Continue-As-New — Composing and Scaling Workflows

When workflows get complex, you need to break them into smaller pieces. When they run forever, you need to manage history growth. This notebook shows how to split work into child workflows, run many children in parallel, and reset long-running workflows with Continue-As-New.

## Learning Objectives

- Use child workflows to decompose complex orchestration
- Understand parent close policies (TERMINATE, ABANDON, REQUEST_CANCEL)
- Fan-out/fan-in with parallel child workflows
- Use Continue-As-New to prevent history from growing unbounded
- Build entity workflows that run indefinitely


## 🛠️ Setup

Make sure Temporal is running:
```bash
cd 03-technologies/workflow-engines/temporal
docker compose up -d
```

### Kernel Selection
Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).
If it doesn't appear, reload the window: `Cmd+Shift+P` → 'Reload Window'.


In [ ]:
import asyncio
import uuid
from datetime import timedelta
from dataclasses import dataclass
from temporalio import activity, workflow
from temporalio.client import Client
from temporalio.worker import Worker
from temporalio.common import RetryPolicy

client = await Client.connect("localhost:7233")
TASK_QUEUE = "advanced-task-queue"
print("✅ Connected to Temporal")


---
## 👶 What is a Child Workflow?

A **child workflow** is a workflow started by another workflow.

Think of the parent as a manager and the children as specialist teams. Each child has its **own event history**, **its own retries**, and **its own lifecycle**. That means one child can fail or retry without forcing the whole parent to restart.

```text
Parent Workflow
  ├── execute_child_workflow(OrderProcessing, order_1)
  ├── execute_child_workflow(OrderProcessing, order_2)
  └── execute_child_workflow(OrderProcessing, order_3)
```

This is useful when one big workflow would otherwise become hard to understand, hard to retry, or too large in history size.


In [ ]:
@activity.defn
async def validate_order(order_id: str) -> str:
    activity.logger.info(f"Validating {order_id}")
    await asyncio.sleep(0.5)
    return "validated"


@activity.defn
async def fulfill_order(order_id: str) -> str:
    activity.logger.info(f"Fulfilling {order_id}")
    await asyncio.sleep(0.5)
    return "shipped"


print("✅ Defined activities: validate_order, fulfill_order")


In [ ]:
@workflow.defn
class ProcessOrderWorkflow:
    @workflow.run
    async def run(self, order_id: str) -> str:
        result = await workflow.execute_activity(
            validate_order,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        result2 = await workflow.execute_activity(
            fulfill_order,
            order_id,
            start_to_close_timeout=timedelta(seconds=10),
        )
        return f"Order {order_id}: {result} → {result2}"


print("✅ Defined child workflow: ProcessOrderWorkflow")


### Practical Exercise: sequential child workflows

We'll start with the simplest version: a parent workflow that launches one child at a time.

This is called **sequential orchestration**:
- the parent starts child 1 and waits
- then starts child 2 and waits
- then starts child 3 and waits

It is easy to reason about, but it is slower because the children do not overlap.


In [ ]:
@workflow.defn
class BatchOrderWorkflow:
    @workflow.run
    async def run(self, order_ids: list[str]) -> list[str]:
        results = []
        for order_id in order_ids:
            result = await workflow.execute_child_workflow(
                ProcessOrderWorkflow.run,
                order_id,
                id=f"order-{order_id}-{workflow.info().workflow_id}",
            )
            results.append(result)
        return results


print("✅ Defined parent workflow: BatchOrderWorkflow")


In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def run_sequential_batch() -> list[str]:
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[BatchOrderWorkflow, ProcessOrderWorkflow],
        activities=[validate_order, fulfill_order],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        return await client.execute_workflow(
            BatchOrderWorkflow.run,
            ["order-1", "order-2", "order-3"],
            id=f"batch-sequential-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


sequential_results = await run_sequential_batch()
print("📦 Sequential child workflow results:")
for item in sequential_results:
    print(f"   • {item}")


---
## ⚡ Fan-Out / Fan-In with Parallel Child Workflows

Sometimes the parent does not need to wait for one child before starting the next.

In that case, the parent can **fan out** by starting many child workflows, and then **fan in** by waiting for all of them to finish.

```text
Parent
  ├── Child A  ┐
  ├── Child B  ├─ run in parallel
  └── Child C  ┘
        ↓
   gather all results
```

This pattern is great for batch work, parallel enrichment, and any case where each piece is independent.


In [ ]:
@workflow.defn
class ParallelBatchWorkflow:
    @workflow.run
    async def run(self, order_ids: list[str]) -> list[str]:
        tasks = [
            workflow.execute_child_workflow(
                ProcessOrderWorkflow.run,
                oid,
                id=f"parallel-{oid}-{workflow.info().workflow_id}",
            )
            for oid in order_ids
        ]
        return list(await asyncio.gather(*tasks))


print("✅ Defined ParallelBatchWorkflow")


In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def run_parallel_batch() -> list[str]:
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[ParallelBatchWorkflow, ProcessOrderWorkflow],
        activities=[validate_order, fulfill_order],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        return await client.execute_workflow(
            ParallelBatchWorkflow.run,
            ["order-4", "order-5", "order-6"],
            id=f"batch-parallel-{uuid.uuid4()}",
            task_queue=TASK_QUEUE,
        )


parallel_results = await run_parallel_batch()
print("⚡ Parallel child workflow results:")
for item in parallel_results:
    print(f"   • {item}")
print()
print("💡 The result shape is the same as the sequential version.")
print("   The difference is that the work can overlap in time.")


---
## 🧭 Parent Close Policies

A **parent close policy** answers one question:

> What should happen to child workflows if the parent closes first?

Temporal gives you three choices:

| Policy | Meaning |
|---|---|
| `TERMINATE` | Stop the child immediately when the parent closes |
| `ABANDON` | Let the child keep running even though the parent is gone |
| `REQUEST_CANCEL` | Ask the child to cancel itself gracefully |

Use `TERMINATE` when the child is meaningful only with the parent. Use `ABANDON` when the child can finish on its own. Use `REQUEST_CANCEL` when you want cleanup logic to run inside the child.


In [ ]:
from temporalio.workflow import ParentClosePolicy

print("ParentClosePolicy values:")
for policy in ParentClosePolicy:
    print(f"   • {policy.name}")


@workflow.defn
class ParentPolicyDemoWorkflow:
    @workflow.run
    async def run(self, order_id: str) -> str:
        return await workflow.execute_child_workflow(
            ProcessOrderWorkflow.run,
            order_id,
            id=f"policy-demo-{order_id}-{workflow.info().workflow_id}",
            parent_close_policy=ParentClosePolicy.ABANDON,
        )


print()
print("✅ Defined ParentPolicyDemoWorkflow using ParentClosePolicy.ABANDON")


### Practical Exercise

Try changing `ParentClosePolicy.ABANDON` to:
- `ParentClosePolicy.TERMINATE`
- `ParentClosePolicy.REQUEST_CANCEL`

Then run the workflow and inspect the Temporal UI. You will see that the parent-child relationship stays the same, but the shutdown behavior changes.


---
## ♻️ Continue-As-New

### The problem

A workflow that loops forever keeps collecting more and more history.

Temporal histories are not supposed to grow without bound. Once a workflow becomes very long-lived, replay gets slower and eventually you can hit history limits.

A common rule of thumb is that very busy workflows should periodically reset themselves before history gets too large.

### The solution

`workflow.continue_as_new()` closes the current run and immediately starts a **fresh run in the same workflow chain**.

```text
Same Workflow ID
run-1 history: [events...events...events]  --continue_as_new-->  run-2 history: [fresh start]
```

The workflow keeps its logical identity, but starts with a clean history.


In [ ]:
@workflow.defn
class EntityCounterWorkflow:
    def __init__(self):
        self._count = 0
        self._signals_since_can = 0

    @workflow.signal
    def increment(self, amount: int = 1):
        self._count += amount
        self._signals_since_can += 1

    @workflow.query
    def get_count(self) -> int:
        return self._count

    @workflow.run
    async def run(self, initial_count: int = 0, force_continue_after: int = 5) -> None:
        self._count = initial_count
        while True:
            await workflow.wait_condition(lambda: self._signals_since_can > 0)
            should_continue = workflow.info().is_continue_as_new_suggested()
            should_continue = should_continue or self._signals_since_can >= force_continue_after
            if should_continue:
                workflow.continue_as_new(self._count, force_continue_after)
            self._signals_since_can = 0
            await asyncio.sleep(1)


print("✅ Defined EntityCounterWorkflow")


### Practical Exercise: entity workflow with signals and queries

An **entity workflow** models one long-lived thing: one user cart, one bank account, one IoT device, or one game room. Instead of finishing quickly, it stays alive and reacts to signals over time.

That makes Continue-As-New a perfect companion:
- the workflow identity stays stable
- the current state is carried forward
- the history stays small enough to replay efficiently


In [ ]:
# NOTE: Temporal normally runs workflow code in a sandbox that re-imports the
# defining module. In a notebook that module is `__main__`, so the re-import
# re-runs these cells and fails with "Failed validating workflow ..." (caused by
# "asyncio.run() cannot be called from a running event loop"). Notebooks must use
# UnsandboxedWorkflowRunner. In a real worker process keep the default sandbox --
# it is what protects you from non-deterministic imports.
from temporalio.worker import UnsandboxedWorkflowRunner
async def demo_entity_workflow() -> None:
    async with Worker(
        client,
        task_queue=TASK_QUEUE,
        workflows=[EntityCounterWorkflow],
        activities=[],
        workflow_runner=UnsandboxedWorkflowRunner(),
    ):
        workflow_id = f"entity-counter-{uuid.uuid4()}"
        handle = await client.start_workflow(
            EntityCounterWorkflow.run,
            # More than one workflow argument must go through args=[...]:
            # start_workflow() accepts only a single positional `arg`.
            args=[10, 3],
            id=workflow_id,
            task_queue=TASK_QUEUE,
        )

        print(f"Started entity workflow: {workflow_id}")

        await handle.signal(EntityCounterWorkflow.increment, 2)
        await asyncio.sleep(0.2)
        print(f"After signal 1, count = {await handle.query(EntityCounterWorkflow.get_count)}")

        await handle.signal(EntityCounterWorkflow.increment, 3)
        await asyncio.sleep(0.2)
        print(f"After signal 2, count = {await handle.query(EntityCounterWorkflow.get_count)}")

        await handle.signal(EntityCounterWorkflow.increment, 5)
        await asyncio.sleep(1.5)

        new_handle = client.get_workflow_handle_for(EntityCounterWorkflow.run, workflow_id)
        print(f"After signal 3, count = {await new_handle.query(EntityCounterWorkflow.get_count)}")
        print("💡 We forced Continue-As-New after 3 signal batches for demo purposes.")

        await new_handle.cancel()
        try:
            await new_handle.result()
        except Exception as exc:
            print(f"Stopped demo workflow: {type(exc).__name__}")


await demo_entity_workflow()


## 🎓 What You Learned

- A **child workflow** is a workflow started by another workflow
- Child workflows keep orchestration modular because each child has its own history and retry behavior
- **Sequential** children are simple to understand, while **parallel** children use fan-out/fan-in for better throughput
- **Parent close policies** decide what happens to children if the parent closes first
- **Continue-As-New** keeps long-lived workflows healthy by resetting history without losing logical identity
- **Entity workflows** are a natural fit for signals, queries, and Continue-As-New

### Suggested next experiments

1. Add retries to `ProcessOrderWorkflow`
2. Make one child fail and observe how the parent reacts
3. Increase the number of signals sent to `EntityCounterWorkflow`
4. Inspect the workflow chain in the Temporal UI and notice the new Run ID after Continue-As-New
